In [2]:
import os
from langchain.chat_models import ChatOpenAI
from IPython.display import display, Markdown, HTML

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings("ignore")

In [3]:

llm = ChatOpenAI(temperature=0)

In [4]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

date_schema = ResponseSchema(name="date",
                             type="date",
                             description="The published date of the news article in date format.")

title_schema = ResponseSchema(name="title",
                              type="string",
                              description="The title of the news article. ")

topic_schema = ResponseSchema(name="topic",
                              type="list",
                              description="The topic of the news article. ")

entity_schema = ResponseSchema(name="entity",
                                    type="list",
                                    description="The entity of the news article. ")

key_words_schema = ResponseSchema(name="key_words",
                                    type="list",
                                    description="The key words of the news article. It should avoid the repetition of the topic and entity.")

lead_paragraph_schema = ResponseSchema(name="lead_paragraph",
                                       type="string",
                                       description="The lead paragraph of the news article. ")


response_schemas = [date_schema,
                    title_schema,
                    topic_schema,
                    entity_schema,
                    key_words_schema,
                    lead_paragraph_schema]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()


In [5]:
from langchain.prompts import ChatPromptTemplate

prompt_template = """\
You are an experienced New York Times journalist, adept at crafting in-depth, well-researched short articles around specific entities or themes. 
Your writing is clear, concise, and engaging, upholding the highest journalistic standards.

Time Range: {time_range}

Entities: {entities}

Themes: {themes}

You are writing a short article about at least one of the entities or themes above. Multiple entities and themes can be included in a single article.

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=prompt_template)

time_range = "2020-1-1 to 2022-12-31"

entities = """
Donald Trump, 
Elon Musk, 
Joe Biden
"""

themes = "The 2020 US Presidential Election, Social Media, The Stock Market"

messages = prompt.format_messages(
    time_range=time_range,
    entities=entities,
    themes=themes,
    format_instructions=format_instructions
)

In [6]:
display(Markdown(messages[0].content))

You are an experienced New York Times journalist, adept at crafting in-depth, well-researched short articles around specific entities or themes. 
Your writing is clear, concise, and engaging, upholding the highest journalistic standards.

Time Range: 2020-1-1 to 2022-12-31

Entities: 
Donald Trump, 
Elon Musk, 
Joe Biden


Themes: The 2020 US Presidential Election, Social Media, The Stock Market

You are writing a short article about at least one of the entities or themes above. Multiple entities and themes can be included in a single article.

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"date": date  // The published date of the news article in date format.
	"title": string  // The title of the news article. 
	"topic": list  // The topic of the news article. 
	"entity": list  // The entity of the news article. 
	"lead_paragraph": string  // The lead paragraph of the news article. 
}
```


In [ ]:
response = llm(messages)
output_dict = output_parser.parse(response.content)
print(output_dict)

In [46]:
display(Markdown(response.content))

```json
{
	"date": "2021-01-20",
	"title": "Joe Biden Inaugurated as 46th President of the United States",
	"topic": ["The 2020 US Presidential Election"],
	"entity": ["Joe Biden"],
	"lead_paragraph": "Joe Biden was sworn in as the 46th President of the United States on Wednesday, January 20, 2021. In his inaugural address, President Biden emphasized the need for unity and healing in a deeply divided nation. He pledged to work tirelessly to tackle the ongoing COVID-19 pandemic, rebuild the economy, address racial injustice, and combat climate change. The inauguration ceremony was held under tight security measures following the storming of the Capitol on January 6, 2021."
}
```

You are a seasoned journalist at the New York Times, specializing in unraveling intricate narratives within complex stories and mapping out the storylines. Your articles should be sharply focused on specific entities such as individuals or events, and themes. We encourage the exploration of interwoven, non-linear storylines encompassing multiple entities and themes.

Time Range: 2020-1-1 to 2022-12-31

Entities: Donald Trump, Elon Musk, Joe Biden

Themes: The 2020 US Presidential Election, Social Media, The Stock Market

Your task is to create a JSON object based on the entities and themes provided above. The composite key in this object should consist of 'TimeFrame' (formatted as "yyyy-mm-dd") and 'Theme'. The value for each key should be an array containing the entities (people, locations, events) that are relevant to the corresponding timeframe and theme. If there are no significant news items related to a particular theme in a specific timeframe, feel free to set the value as an empty array.